# Neural Data Clustering Analysis

**Author:** Parviz Ghaderi  
**Date:** 2025  
**Manuscript:** "Contextual gating of whisker-evoked responses by frontal cortex supports flexible decision making"

## Description
This notebook performs comprehensive unsupervised clustering analysis on neural data for the manuscript:  
_"Contextual gating of whisker-evoked responses by frontal cortex supports flexible decision making"_

The analysis pipeline includes:
- **PCA Analysis**: Identifies significant principal components using permutation testing
- **Spectral Embedding**: Transforms data into spectral space for optimal cluster separation
- **Gaussian Mixture Model Clustering**: Applies GMM with grid search for optimal parameters
- **Cluster Validation**: Uses BIC criteria and hierarchical clustering for validation
- **Results Export**: Saves clustering results in multiple formats for further analysis

## Scientific Objectives
- Identify functionally distinct neural populations in the recorded data
- Determine optimal clustering parameters using statistical criteria
- Analyze the relationship between brain areas, cell types, and cluster membership
- Provide quantitative measures of cluster quality and stability

## Methodology Overview
1. **Dimensionality Reduction**: PCA identifies significant components via permutation testing
2. **Spectral Embedding**: Transforms PCA space into spectral coordinates preserving local structure
3. **Model Selection**: Grid search finds optimal GMM parameters using BIC
4. **Validation**: Hierarchical clustering confirms cluster robustness
5. **Export**: Results saved in multiple formats for downstream analysis

## Requirements
```bash
pip install numpy matplotlib scipy scikit-learn pandas joblib
```

## Instructions
1. Ensure the data files `Neuronal_data_100ms.mat` and `Neuronal_data_10ms_withroc.mat` are in the current directory
2. Run all cells in order
3. Results will be saved as `Data_Clustering.mat` and `Data_Clustering.pkl`

## Expected Outputs
- **Data_Clustering.mat**: MATLAB-compatible results file
- **Data_Clustering.pkl**: Python pickle file with complete results
- **GMM_100ms_Run1.pkl**: GMM model results and parameters
- **spectral_Data.npz**: Spectral embedding coordinates
- **pca_analysis_data.npz**: PCA analysis results



## 1. Import Libraries and Set Parameters


In [ ]:
# Import required libraries
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.io import loadmat
from scipy.stats import ttest_1samp
from scipy.spatial.distance import pdist, squareform
from scipy.linalg import eigh
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import pairwise_distances
import joblib
import os
from scipy.spatial.distance import pdist, squareform as scipy_squareform
from scipy.linalg import eigh
from sklearn.decomposition import PCA
from scipy.stats import ttest_1samp
# Set matplotlib backend and parameters for publication-quality figures
matplotlib.use('TkAgg')
matplotlib.rcParams['pdf.fonttype'] = 42  # Embed fonts as editable text
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.linewidth'] = 0.5
matplotlib.rcParams['xtick.major.width'] = 0.5
matplotlib.rcParams['ytick.major.width'] = 0.5
matplotlib.rcParams['figure.dpi'] = 300
matplotlib.rcParams['savefig.dpi'] = 300

# Set random seed for reproducibility
np.random.seed(42)

print("All packages imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")


## 2. Define Core Analysis Functions

The following functions implement the main algorithms for PCA analysis, spectral embedding, and clustering. Each function is designed to be modular and reusable, with comprehensive documentation and error handling.


In [ ]:

def pca_analysis(neural_data, plot_ind, path, seed):
    """
    Perform PCA analysis on neural data with permutation testing to determine significant PCs.

    Args:
        neural_data (numpy.ndarray): Neural data matrix (neurons x time bins).
        plot_ind (bool): If True, generate and display plots.
        path (str): Path to save PCA analysis data.
        seed (int): Random seed for reproducibility.

    Returns:
        int: Number of significant principal components.
    """
    np.random.seed(0)
    centered = False

    # Mean-center the data (rows = neurons, columns = time bins)
    temp_nd = neural_data - neural_data.mean(axis=1, keepdims=True)

    # Apply PCA
    pca = PCA()
    pca.fit(temp_nd.T)
    score = pca.transform(temp_nd.T)
    latent = pca.explained_variance_  # Variances of principal components

    # Plot explained variance ratios
    if plot_ind:
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.plot(np.cumsum(latent) / np.sum(latent))
        plt.grid()
        plt.title("Cumulative Variance Ratio")

        plt.subplot(1, 2, 2)
        plt.plot(latent / np.sum(latent))
        plt.grid()
        plt.title("Variance Ratio")
        plt.show(block=False)

        # Plot first 80 principal components
        num_kernels = min(80, score.shape[1])
        num_rows = int(np.ceil(num_kernels / 6))
        plt.figure(figsize=(15, num_rows * 2))
        for i in range(num_kernels):
            plt.subplot(num_rows, 6, i + 1)
            plt.plot(score[:, i])
            plt.title(f"Kernel {i + 1}")
            plt.grid()
            plt.xticks([])
            plt.yticks([])
        plt.tight_layout()
        plt.show(block=False)

    # Null distribution via permutation testing
    null_sample = 1000
    null_sigma = np.zeros((null_sample, len(latent)))
    for j in range(null_sample):
        temp = np.copy(temp_nd)
        for i in range(temp.shape[0]):
            np.random.shuffle(temp[i, :])
        null_pca = PCA()
        null_pca.fit(temp.T)
        null_sigma[j, :] = null_pca.explained_variance_
        if j % 100 == 0:
            print(f"Progress: {j}/{null_sample} iterations completed")
    # Save null distribution
    np.savez(f"{path}/pca_analysis_data.npz", null_sigma=null_sigma, latent=latent)

    # Compute mean and standard deviation of null distribution
    y = np.mean(null_sigma, axis=0)
    dy = np.std(null_sigma, axis=0)

    # Plot comparison with null distribution
    if plot_ind:
        plt.figure()
        plt.plot(latent, linewidth=2, color="#4B3F72", label="Variance Ratio")
        plt.plot(y, linewidth=2, color="#0072BD", label="Null Variance Ratio")
        plt.fill_between(range(len(y)), y - dy, y + dy, color="#0072BD", alpha=0.3, linestyle="--")
        plt.grid()
        plt.legend()
        plt.title("Variance Ratio vs Null Distribution")
        plt.show(block=False)

    # Significance testing with Bonferroni correction
    p_vals = np.array([ttest_1samp(null_sigma[:, i], latent[i])[1] for i in range(len(latent))])
    significant_pcs = np.where((latent > y) & (p_vals < 0.05 / len(latent)))[0]

    return len(significant_pcs)

def squareform(Y, dir=None):
    """
    
    Reformat a distance matrix between upper triangular and square form.

    Args:
        Y (ndarray): Input vector or matrix.
        dir (str): 'tovector' or 'tomatrix' to force the format.

    Returns:
        ndarray: Reformatted distance matrix.
    """
    np.random.seed(0)
    if dir is None:
        dir = 'tomatrix' if Y.ndim == 1 else 'tovector'

    if dir == 'tomatrix':
        return scipy_squareform(Y)
    elif dir == 'tovector':
        return scipy_squareform(Y)
    else:
        raise ValueError("Invalid direction specified.")

def spect_clust(S, K):
    """
    Spectral clustering helper function.

    Args:
        S (ndarray): Similarity matrix.
        K (int): Number of clusters (-1 for all eigenvalues).

    Returns:
        tuple: Eigenvalues, eigenvectors, and flag.
    """
    np.random.seed(0)
    DinvS = np.diag(1.0 / np.sqrt(S.sum(axis=1)))
    L = np.eye(S.shape[0]) - DinvS @ S @ DinvS

    if K == -1:
        Lambda, V = eigh(L)
        flag = "All"
    else:
        Lambda, V = eigh(L, subset_by_index=[0, K - 1])
        flag = "SmallestAbs"

    Lambda = np.sort(Lambda)

    plt.figure()
    plt.plot(np.diff(Lambda[1:40]), '*')
    plt.title("Eigenvalue Differences")
    plt.show(block=False)

    return Lambda, V, flag

def A4_spectral_embedding(Neural_Data, N_PCA, plot_ind, Path, Seed):
    """
    Perform spectral embedding on neural data.

    Args:
        Neural_Data_5T2P_preprocessed (dict): Preprocessed neural data.
        N_PCA (int): Number of principal components.
        plot_ind (bool): Indicator to enable/disable plots.
        Path (str): Path to save results.
        Seed (int): Random seed for reproducibility.

    Returns:
        dict: Spectral data and cluster information.
    """
    np.random.seed(Seed)

    temp_ND_pdf = Neural_Data['Neurons_activity_condition_area']
    temp_ND_area = Neural_Data['Area']
    sig = 0.07975 #  0.07975 for 100 ms             and 0.1065 for 10 ms
    K_Spect = 17  # From elbow method 16 for 100 ms and 31 for 10 ms
    print("Number of neurons:", temp_ND_pdf.shape[0])           # Number of neurons: 10294
    # PCA
    temp_ND_pdf = temp_ND_pdf - np.mean(temp_ND_pdf, axis=1, keepdims=True)
    PCA_Data= np.linalg.svd(temp_ND_pdf, full_matrices=False)[0][:, :N_PCA]
    
    print("Number of PCA_Data:", PCA_Data.shape[0])
    D_Both = squareform(pdist(PCA_Data, metric='euclidean'))
    S_Both = np.exp(-D_Both / sig)
    print("Average similarity:", np.mean(S_Both))

    Lambda_Both, V_Both, flag = spect_clust(S_Both, -1)
    print("Spectral embedding flag:", flag)

    spectral_Data = V_Both[:, 1:K_Spect]  # Exclude the first eigenvector


    Spectral_Data = {
        'spectral_Data': spectral_Data,
        'ClusterCounter': Neural_Data ['clustercounter']
    }

    np.savez(f"{Path}/Spectral_Data", Spectral_Data=Spectral_Data, sig=sig)

    if plot_ind:
        plt.figure()
        plt.hist(S_Both.flatten(), bins=50, color='blue', alpha=0.7)
        plt.grid()
        plt.xlim([0, 1])
        plt.title('Similarity Histogram')
        plt.show(block=False)

        N_Neurons = temp_ND_pdf.shape[0]
        areas = np.unique(temp_ND_area)
        Ind = np.arange(N_Neurons)
        Ind2 = []

        plt.figure()
        plt.imshow(S_Both, aspect='auto', cmap='viridis')

        for area in areas:
            temp = np.max(Ind[temp_ND_area.flatten() == area])
            Ind2.append(int(np.median(np.where(temp_ND_area == area)[0])))
            plt.plot([0, N_Neurons], [temp, temp], 'k-', linewidth=0.5)
            plt.plot([temp, temp], [0, N_Neurons], 'k-', linewidth=0.5)

        plt.colorbar()
        plt.title("Similarity Heatmap")
        plt.xticks(Ind2, areas, rotation=90)
        plt.yticks(Ind2, areas)
        plt.show(block=False)

        plt.figure()
        plt.plot(Lambda_Both[1:], '*')
        plt.title("Eigenvalues")
        plt.axis([1, 50, 0.96, 1])
        plt.grid()
        plt.show(block=False)

        plt.figure()

        plt.scatter(V_Both[:, 0], V_Both[:, 1],V_Both[:, 2])
        plt.title("Neurons in Spectral Space")
        plt.grid()
        plt.show(block=False)

    return Spectral_Data

import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
import joblib

def A6_GMM_grid_search(Spectral_Data, Path, Seed,Run):
    """
    Perform a grid search to find the best Gaussian Mixture Model (GMM) based on BIC.
    
    Args:
        Spectral_Data (dict): Dictionary with 'spectral_Data' containing the dataset.
        Path (str): Path to save the results.
        Seed (int): Random seed for reproducibility.
    
    Returns:
        GaussianMixture: The best GMM model based on BIC.
    """
    np.random.seed(Seed)
    
    X = Spectral_Data['spectral_Data']
    
    k = list(range(10, 25, 5)) + list(range(25, 45, 1)) + list(range(45, 56, 5))  # Number of components to test
    Sigma = ['diag']  # Covariance types to test   diagonal or full
    SharedCovariance = [False]  # Shared covariance (not directly in sklearn)
    RegularizationValue = 0  # Regularization for GMM
    MaxIter = 1000
    Replicates =1000  # Number of initializations

    nK = len(k)
    nSigma = len(Sigma)
    nSC = len(SharedCovariance)
    
    # Preallocate results
    gm_models = []
    aic = np.zeros((nK, nSigma, nSC))
    bic = np.zeros((nK, nSigma, nSC))
    converged = np.zeros((nK, nSigma, nSC), dtype=bool)
    
    results = {
        "Model": np.empty((nK, nSigma, nSC), dtype=object),
        "AIC": np.zeros((nK, nSigma, nSC)),
        "BIC": np.zeros((nK, nSigma, nSC)),
        "SharedCov": np.empty((nK, nSigma, nSC), dtype=object),
        "Rep": np.full((nK, nSigma, nSC), Replicates),
        "Covtype": np.empty((nK, nSigma, nSC), dtype=object),
        "MaxIte": np.full((nK, nSigma, nSC), MaxIter),
        "Reg": np.full((nK, nSigma, nSC), RegularizationValue),
        "k": np.zeros((nK, nSigma, nSC)),
    }
    
    for m, shared in enumerate(SharedCovariance):
        for j, cov_type in enumerate(Sigma):
            for i, n_components in enumerate(k):
                print(f"Training GMM with k={n_components}, CovType={cov_type}, SharedCov={shared}")
                gmm = GaussianMixture(
                    n_components=n_components,
                    covariance_type=cov_type,
                    max_iter=MaxIter,
                    reg_covar=RegularizationValue,
                    random_state=Seed,
                    n_init=Replicates
                )
                
                Y=np.array(X)
                gmm.fit(Y)
                
                # Save model and metrics
                gm_models.append(gmm)
                aic[i, j, m] = gmm.aic(X)
                bic[i, j, m] = gmm.bic(X)
                converged[i, j, m] = gmm.converged_
                
                results["Model"][i, j, m] = gmm
                results["AIC"][i, j, m] = aic[i, j, m]
                results["BIC"][i, j, m] = bic[i, j, m]
                results["SharedCov"][i, j, m] = shared
                results["Covtype"][i, j, m] = cov_type
                results["k"][i, j, m] = n_components
    
    # Save results
    joblib.dump(results, f"{Path}/GMM_100ms_Run{str(Run)}.pkl")
    
    all_converged = np.all(converged)
    print(f"All GMM models converged: {all_converged}")
    
    # Plot BIC values
    plt.figure()
    for j in range(nSigma * nSC):
        plt.plot(k, bic[:, j, 0], marker='o', label=f"Model {j}")
    plt.title("BIC for Various k and Covariance Choices")
    plt.xlabel("Number of Components (k)")
    plt.ylabel("BIC")
    plt.legend()
    plt.show(block=False)
    
    # Find the best model
    best_index = np.unravel_index(np.argmin(bic), bic.shape)
    best_gmm = results["Model"][best_index]
    
    return best_gmm


## 3. Load and Explore Data

Before performing clustering analysis, we need to load the neural data and understand its structure. This section loads the data files and provides initial exploration to understand the dataset characteristics.


In [ ]:
# Load neural data and examine structure
print("Loading neural data...")

# Load the main neural data file
Data = loadmat('../processed_data/Neuronal_data_100ms.mat')
print(Data.keys())
# Display the structure of the loaded data
print("\nData structure:")
print("=" * 50)
for key in Data.keys():
    if not key.startswith('__'):  # Skip internal MATLAB variables
        if isinstance(Data[key], np.ndarray):
            print(f"{key}: {Data[key].shape} - {Data[key].dtype}")
        else:
            print(f"{key}: {type(Data[key])}")

# Extract key variables
neural_activity = Data['Neurons_activity_condition_area']
brain_areas = Data['Area']
neuron_types = Data['Type']
depths = Data['Depth']
layers = Data['Layer']
cluster_counter = Data['clustercounter']
unit_ids = Data['Unit_ids']


print(f"\nDataset Summary:")
print(f"Neural activity matrix: {neural_activity.shape}")
print(f"Number of neurons: {neural_activity.shape[0]}")
print(f"Number of time bins: {neural_activity.shape[1]}")
print(f"Brain areas: {np.unique(brain_areas)}")
print(f"Neuron types: {np.unique(neuron_types)}")

### 3. PCA Analysis and Extraction of Significant Principal Components

In this cell, we will perform PCA analysis on the neural data and extract the number of significant principal components (PCs). The PCA analysis involves mean-centering the data, applying PCA, and performing permutation testing to determine the significance of each PC. The number of significant PCs will be returned as the output.

In [ ]:
significant_pcs = pca_analysis(Data['Neurons_activity_condition_area'], plot_ind=False, path="../processed_data/", seed=0)
# significant_pcs=17
print("Number of significant PCs:", significant_pcs)

## 4. Perform Spectral Embedding Analysis

Now we perform the spectral embedding analysis. This step transforms the neural data into a spectral space where similar neurons are close together, making clustering more effective.


In [ ]:
# Perform spectral embedding analysis
print("Starting spectral embedding analysis...")

# Parameters for spectral embedding
# N_PCA = 17  # Number of principal components for 100ms resolution
N_PCA = significant_pcs
plot_ind = True  # Generate plots
Path = "../processed_data/"  # Save path
Seed = 0  # Random seed

# Run spectral embedding
spectral_data = A4_spectral_embedding(Data, N_PCA, plot_ind, Path, Seed)

print("Spectral embedding analysis completed!")
print(f"Spectral data shape: {spectral_data['spectral_Data'].shape}")

## 5. Gaussian Mixture Model Clustering

The next step is to perform Gaussian Mixture Model (GMM) clustering on the spectral embedding. We use a grid search approach to find the optimal number of clusters based on the Bayesian Information Criterion (BIC).


In [ ]:
# Run GMM grid search
print("Starting GMM clustering analysis...")

# Parameters for GMM analysis
Path = "../processed_data/"  # Save path
Seed = 0  # Random seed
Run = 0  # Run identifier

# Run GMM grid search
GMModel = A6_GMM_grid_search(spectral_data, Path=Path, Seed=Seed, Run=Run)

print("GMM clustering analysis completed!")
print(f"Best model has {GMModel.n_components} components")


## 6. Cluster Analysis and Validation

After determining the optimal number of clusters, we perform additional validation using hierarchical clustering and analyze the cluster assignments.


In [ ]:

# Load GMM results and perform cluster validation
print("Loading GMM results for cluster validation...")

# Load the GMM results
results = joblib.load(r"..\processed_data\GMM_100ms_Run0.pkl")
# Load using np.load
data = np.load(r"..\processed_data\Spectral_Data.npz", allow_pickle=True)
spectral_data = data['Spectral_Data'].item()  # .item() converts back to dict# Extract the best model
best_index = np.unravel_index(np.argmin(results["BIC"]), results["BIC"].shape)
best_gmm = results["Model"][best_index]

print(f"Best model: {int(results['k'][best_index])} components")
print(f"BIC: {results['BIC'][best_index]:.2f}")

# Get cluster means and perform hierarchical clustering
cluster_means = best_gmm.means_
print(f"Cluster means shape: {cluster_means.shape}")

# Calculate pairwise distances between cluster means
distances = pairwise_distances(cluster_means)
print(f"Distance matrix shape: {distances.shape}")

# Perform hierarchical clustering
Z = linkage(distances, method='ward')

# Determine optimal number of clusters using dendrogram
max_d = 0.032  # Maximum distance threshold
clusters = fcluster(Z, max_d, criterion='distance')
num_clusters = len(np.unique(clusters))

print(f"Optimal number of clusters (hierarchical): {num_clusters}")

# Plot dendrogram
plt.figure(figsize=(3, 3))
dendrogram(Z, labels=np.arange(1, len(cluster_means) + 1))
plt.axhline(y=max_d, color='r', linestyle='--', linewidth=2, 
           label=f'Cutoff at {max_d}')
plt.title('Dendrogram for GMM Cluster Validation', fontsize=10, fontweight='bold')
plt.xlabel('Cluster Index')
plt.ylabel('Distance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show(block=False)

print(f"Hierarchical clustering validation completed!")


# Create a single figure and axes to overlay both runs
plt.figure(figsize=(3, 3))
# 3. Extract and sort BIC values
BIC_3D = results["BIC"]
k_3D = results["k"]
BIC_1D = BIC_3D[:, 0, 0]
k_vals_1D = k_3D[:, 0, 0]
indices_sorted = np.argsort(k_vals_1D)
sorted_k = k_vals_1D[indices_sorted]
sorted_BIC = BIC_1D[indices_sorted]

# 4. Find the minimum and plot
min_bic_idx = np.argmin(sorted_BIC)
min_bic_k = sorted_k[min_bic_idx]
min_bic_val = sorted_BIC[min_bic_idx]

plt.plot(sorted_k, sorted_BIC, '-o', color='b', label=f'Run {1}')
plt.plot(min_bic_k, min_bic_val, 's', color='b', ms=5)
plt.annotate(
    f"Min BIC: {min_bic_val:.2f}\nat k={int(min_bic_k)}",
    xy=(min_bic_k, min_bic_val),
    xytext=(min_bic_k + 5, min_bic_val+1000),
    arrowprops=dict(arrowstyle="->", color='k'),
    ha='left',
    color='k'
)

plt.xlabel('Number of Components (k)')
plt.ylabel('BIC')
plt.title('BIC vs. Number of GMM Components (Run0)')
plt.legend(loc='best')
plt.tick_params(direction='out', top=False, right=False)
plt.box(False)
plt.show(block=False)    


In [ ]:
# Export results in multiple formats
print("Exporting clustering results...")
# 1. Identify the index of the model with the lowest BIC
bic_values = results["BIC"].ravel()    # Flatten if needed
q = np.argmin(bic_values)              # q is the index of the smallest BIC
model = results["Model"].ravel()[q]
print(f"The best number of components is {model.n_components}")
X = spectral_data["spectral_Data"]
clusterLabels = model.predict(X)
clusterLabels=clusterLabels+1      # To match the MATLAB indexing
nNeurons = len(clusterLabels)
print(f"Assigned cluster labels (first 10): {list(clusterLabels[:10])}")
print(f"Number of neurons (data points): {nNeurons}")
# Reorder neurons by cluster for better visualization
nClusters = model.n_components
clusterLabels = clusterLabels.astype(int)
# Use clusterLabels directly (already 1ndexed after +1 operation)
# Convert to float to match orinal dtype expectation
newClusterLabels = clusterLabels.astype(float)

# Sort neurons by cluster
ind_temp = np.argsort(newClusterLabels)
T_temp = newClusterLabels[ind_temp]

newclusterTags = [str(i + 1) for i in range(nClusters)]
# load 10ms version od data 
Data = loadmat('../processed_data/Neuronal_data_10ms_withroc.mat')

# Create comprehensive results dictionary
Cl = {
    "Cluster_Counter_Ordered": T_temp,
    "Neural_Data_normalized_Ordered": Data["Neurons_activity_condition_area"][ind_temp, :],
    "Neuron_Counter_Ordered": Data["clustercounter"][ind_temp],
    "Areas_Ordered": Data["Area"][ind_temp],
    "Id_Ordered": Data["Unit_ids"][ind_temp],
    "Type_Ordered": Data["Type"][ind_temp],
    "Depth_Ordered": Data["Depth"][ind_temp],
    "Layer_Ordered": Data["Layer"][ind_temp],
    "CCF_location_Ordered": Data["CCF_location"][ind_temp],
    "CCF_xyz_Ordered": Data["CCF_xyz"][ind_temp],
    "Cluster_Tags": newclusterTags,
    "Selectivity_index": Data["Discrimination_index"][ind_temp]
}

In [ ]:
# Save as MATLAB .mat file
import scipy.io as sio
matlab_dict = {key: np.array(value) for key, value in Cl.items()}
sio.savemat("../processed_data/Data_Clustering.mat", Cl)
print("Data_Clustering.mat saved successfully!")

# Save as Python pickle file
import pickle
with open("../processed_data/Data_Clustering.pkl", "wb") as f:
    pickle.dump(Cl, f)
print("Data_Clustering.pkl saved successfully!")

print("\nClustering analysis completed successfully!")